# Bundle Loading Guide Notebook

Mirrors the [Bundle Loading Guide](https://gongjr0.github.io/SymbolicDSGE/latest/guides/bundle_loading_guide/). Pair with the [authoring notebook](bundle_authoring.ipynb) — the bundle this opens (`experiment-1.sdsge`) is produced there.

In [19]:
import numpy as np

from SymbolicDSGE import DSGESolver, load_bundle
from SymbolicDSGE.estimation.results import MCMCResult
from SymbolicDSGE.estimation.spec import MCMCResultMeta, OptimizationResultMeta
from SymbolicDSGE.monte_carlo import run_pipeline, validate_pipeline_spec

## Open the bundle

In [20]:
loaded = load_bundle("experiment-1.sdsge")

print("Created by:", loaded.manifest.created_by)
print("Created at:", loaded.manifest.created_at)
print("Format version:", loaded.manifest.sdsge_version)

Created by: experiment-1
Created at: 2026-06-14T14:35:39.036609+00:00
Format version: 1


## Reach the re-solved models

In [21]:
reference = loaded.reference
if reference is not None:
    print("Stable:", reference.policy.stab == 0)
    print("Eigenvalues:", reference.policy.eig)
    print("A shape:", reference.A.shape)

Stable: True
Eigenvalues: [0.28018451+0.j 0.83      +0.j 0.85      +0.j 2.60451546+0.j
 1.18546572+0.j]
A shape: (5, 5)


In [22]:
T = 20
rng = np.random.default_rng(42)
shocks = {
    "g,z": rng.standard_normal((T, 2)),
}
sim = reference.sim(
    T=T,
    shocks=shocks,
    observables=True,
)
print(sim["Infl"][:5])

[  3.43         8.20387037  10.83531974 -12.17755328   0.91492503]


## Reach the estimation tab

In [23]:
estimation = loaded.estimation

if estimation is not None:
    print("Method:", estimation.spec.method)
    print("Observables:", estimation.spec.observables)
    print("Parameters:", [p.name for p in estimation.spec.parameters])

Method: mcmc
Observables: ['OutGap', 'Infl', 'Rate']
Parameters: ['psi_pi', 'psi_x']


In [24]:
result = estimation.result
if isinstance(result, OptimizationResultMeta):
    print("Point estimate:", result.theta)
    print("Log-posterior:", result.logpost)
elif isinstance(result, MCMCResultMeta):
    print("Acceptance:", round(result.accept_rate, 2))
    print("Draws:", result.n_draws, "burn-in:", result.burn_in)

Acceptance: 0.34
Draws: 1000 burn-in: 200


In [25]:
if estimation.observed is not None:
    print("Observed shape:", estimation.observed.shape)
if estimation.posterior is not None:
    samples = estimation.posterior["samples"]
    print("Posterior mean:", samples.mean(axis=0))

Observed shape: (40, 3)
Posterior mean: [1.88370835 0.71293637]


Reconstruct a live `MCMCResult` for sampling diagnostics when an MCMC bundle is loaded:

```python
if isinstance(result, MCMCResultMeta) and estimation.posterior is not None:
    mcmc = MCMCResult(
        param_names=result.param_names,
        samples=estimation.posterior["samples"],
        logpost_trace=estimation.posterior["logpost"],
        accept_rate=result.accept_rate,
        n_draws=result.n_draws,
        burn_in=result.burn_in,
        thin=result.thin,
    )
    mcmc.hpd_intervals(alpha=0.05)
```

### Re-run an estimation from a loaded bundle

`spec.to_estimator_inputs()` lowers the loaded spec into the concrete arguments `DSGESolver.estimate` expects — `Prior` objects built from each `PriorSpec`, parameter initials, and (for MLE/MAP) bounds.

In [26]:
# inputs = estimation.spec.to_estimator_inputs()

# solver = DSGESolver(loaded.reference.config, loaded.reference.kalman_config)
# extras: dict = {}
# if inputs.bounds is not None and estimation.spec.method != "mcmc":
#     extras["bounds"] = inputs.bounds  # MLE/MAP only

# fresh_result = solver.estimate(
#     compiled=loaded.reference.compiled,
#     y=estimation.observed,
#     method=estimation.spec.method,
#     estimated_params=inputs.estimated_params,
#     theta0=inputs.theta0,
#     priors=inputs.priors,
#     observables=estimation.spec.observables,
#     **extras,
# )
# print("Re-run result:", type(fresh_result).__name__)

if isinstance(result, MCMCResultMeta) and estimation.posterior is not None:
    mcmc = MCMCResult(
        param_names=result.param_names,
        samples=estimation.posterior["samples"],
        logpost_trace=estimation.posterior["logpost"],
        accept_rate=result.accept_rate,
        n_draws=result.n_draws,
        burn_in=result.burn_in,
        thin=result.thin,
    )
    print(mcmc.hpd_intervals(alpha=0.05))

{'psi_pi': (np.float64(1.4619459318802717), np.float64(2.280162974441619)), 'psi_x': (np.float64(0.36242444759843134), np.float64(1.053770473394954))}


## Reach the Monte Carlo tab

In [27]:
mc = loaded.mc

if mc is not None:
    print("Pipeline nodes:", [n.id for n in mc.spec.nodes])
    print("Pipeline edges:", [(e.source, e.target) for e in mc.spec.edges])

    if mc.document is not None:
        wire = mc.wire()
        print("Run kind:", wire["kind"])

Pipeline nodes: ['sim', 'jb']
Pipeline edges: [('sim', 'jb')]


### Re-run a Monte Carlo pipeline from a loaded bundle

`monte_carlo.run_pipeline` validates the graph against `STEP_CATALOG`, compiles each step, and runs it against the loaded models. No `[ui]` extra is required.

In [43]:
try:
    ordered = validate_pipeline_spec(
        loaded.mc.spec,
        has_reference=loaded.reference is not None,
        has_dgp=loaded.dgp is not None,
    )
    print("Topological order:", [n.id for n in ordered])
except ValueError as e:
    print(
        "We don't have a DGP in the experiment, so the pipeline is invalid.\n"
        "See the cell below."
    )

We don't have a DGP in the experiment, so the pipeline is invalid.
See the cell below.


In [ ]:
# The demo pipeline expects a DGP; substitute loaded.reference when no DGP was bundled
# (typical for the authoring notebook's output, which carries reference only).
dgp_for_run = loaded.dgp if loaded.dgp is not None else loaded.reference

mc_result = run_pipeline(
    loaded.mc.spec,
    reference=loaded.reference,
    dgp=dgp_for_run,
    n_rep=50,
    fail_fast=True,
)
print("Successful reps:", mc_result.n_successful, "/", mc_result.n_rep)

jb = mc_result.test_summaries["Normality Check"]
stat_ci = jb.statistic_confidence_interval(0.95)
pval_ci = jb.pval_confidence_interval(0.95)

list(zip(stat_ci, pval_ci))

Successful reps: 50 / 50


[(np.float64(1.288096955083899), np.float64(0.011038884327619805)),
 (np.float64(2.283301139827742), np.float64(0.1346009068750702))]

## Reach the simulation prefill

In [38]:
sim_spec = loaded.simulation

if sim_spec is not None:
    print("T:", sim_spec.T)
    print("Observables flag:", sim_spec.observables)
    if sim_spec.shock_generation is not None:
        print("Seed:", sim_spec.shock_generation.seed)
        print("Distribution:", sim_spec.shock_generation.dist)

T: 25
Observables flag: True
Seed: 42
Distribution: norm
